In [1]:
import pandas as pd
ar = pd.read_csv('articles.csv')
cus = pd.read_csv('customers.csv')
tra = pd.read_csv('transactions_train.csv')

# 결측치

- detail_desc(416)

In [2]:
ar.isna().sum()

article_id                        0
product_code                      0
prod_name                         0
product_type_no                   0
product_type_name                 0
product_group_name                0
graphical_appearance_no           0
graphical_appearance_name         0
colour_group_code                 0
colour_group_name                 0
perceived_colour_value_id         0
perceived_colour_value_name       0
perceived_colour_master_id        0
perceived_colour_master_name      0
department_no                     0
department_name                   0
index_code                        0
index_name                        0
index_group_no                    0
index_group_name                  0
section_no                        0
section_name                      0
garment_group_no                  0
garment_group_name                0
detail_desc                     416
dtype: int64

- FN(895050)
- Active(907576)
- club_member_status(6062)
- fashion_news_frequency(16011)
- age(15861)

In [3]:
cus.isna().sum()

customer_id                    0
FN                        895050
Active                    907576
club_member_status          6062
fashion_news_frequency     16011
age                        15861
postal_code                    0
dtype: int64

In [4]:
tra.isna().sum()

t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

 # 전처리

- articles.csv
- 분석에 필요없는 컬럼삭제

In [ ]:
columns_to_drop = ['detail_desc','product_code','product_type_no','graphical_appearance_no','colour_group_code','perceived_colour_value_id','perceived_colour_master_id','department_no','index_code','index_group_no','section_no','garment_group_no','index_group_name','index_name','section_name','perceived_colour_master_name']
art_df = ar.drop(columns=columns_to_drop)

- customers.csv
- 분석에 필요없는 컬럼삭제
- age정수로 변환후 age는 대체하기 힘들기때문에 결측치제거

In [ ]:
columns_to_drop = ['postal_code','FN','Active','club_member_status','fashion_news_frequency']
custom_df = cus.drop(columns=columns_to_drop)
custom_df['age'] = custom_df['age'].astype('Int64')
custom_df.dropna(subset=['age'], inplace=True)

- transactions_train.csv
- 590을 곱해 H&M 실제 유로/달러(EUR/USD) 가격으로 복원
- t_dat분석을 위한 타입변환

In [ ]:
tra['price_eur'] = round(tra['price'] * 590, 2)
tra['t_dat'] = pd.to_datetime(tra['t_dat'])

# 이상치

##### 원저라이징

- 하위 1%에 해당하는 낮은 가격들을 삭제하지 않고 모두 하위 1% 기준값으로 통일
- 너무 낮은 가격 때문에 평균이 왜곡되는 것을 막으면서도, 데이터의 양을 손실 없이 그대로 유지

In [ ]:
# 하위 1% 기준값 저장
lower_df = tra['price_eur'].quantile(0.01)
# 하위 1% 미만인 값들을 모두 2.11로 변경
tra['price_eur'] = tra['price_eur'].clip(lower=lower_df)